# Build and Share a Fun ML App to Reveal Someone's Personality by How They Write


This notebook guides you through building a simple, interactive machine learning app using Gradio and Hugging Face Transformers.
The app will take a piece of text, analyze the writing style, and predict the likely MBTI personality type.


## Step 1, Load the Dataset

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
# prompt: upload file from local computer

from google.colab import files
uploaded = files.upload()
for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))


Saving mbti_1.csv to mbti_1 (1).csv
User uploaded file "mbti_1 (1).csv" with length 62856486 bytes


In [14]:

import pandas as pd

filename = 'mbti_1 (1).csv'

df = pd.read_csv(filename)
df['posts'] = df['posts'].apply(lambda x: x.replace('|||', ' '))
df['type'].unique()


array(['INFJ', 'ENTP', 'INTP', 'INTJ', 'ENTJ', 'ENFJ', 'INFP', 'ENFP',
       'ISFP', 'ISTP', 'ISFJ', 'ISTJ', 'ESTP', 'ESFP', 'ESTJ', 'ESFJ'],
      dtype=object)

Exploration of the Data

In [17]:
df[['type', 'posts']].head()

,type,posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw ht...
1,ENTP,'I'm finding the lack of me in these posts ver...
2,INTP,'Good one _____ https://www.youtube.com/wat...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o..."
4,ENTJ,'You're fired. That's another silly misconcept...


In [20]:
df['type']

,type
0,INFJ
1,ENTP
2,INTP
3,INTJ
4,ENTJ
...,...
8670,ISFP
8671,ENFP
8672,INTP
8673,INFP


## Step 2, Understanding the MBTI


The Myers-Briggs Type Indicator categorizes people into 16 types using four axes:
- Introversion or Extroversion
- Intuition or Sensing
- Thinking or Feeling
- Perceiving or Judging


## Step 3, Prepare the Data

In [18]:

from sklearn.model_selection import train_test_split

df = df[df['posts'].notna()]
X_train, X_test, y_train, y_test = train_test_split(df['posts'], df['type'], test_size=0.2, random_state=42)


## Step 4, Train the Transformer Model

In [19]:

!pip install transformers datasets --quiet

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

unique_labels = sorted(df['type'].unique())
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

train_data = pd.DataFrame({'text': X_train, 'label': [label2id[y] for y in y_train]})
test_data = pd.DataFrame({'text': X_test, 'label': [label2id[y] for y in y_test]})

train_dataset = Dataset.from_pandas(train_data)
test_dataset = Dataset.from_pandas(test_data)

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=16,
    id2label=id2label,
    label2id=label2id
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 12.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cu

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Map:   0%|          | 0/6940 [00:00<?, ? examples/s]

Map:   0%|          | 0/1735 [00:00<?, ? examples/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=1,
    evaluation_strategy='epoch',
    logging_dir='./logs',
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()


## Step 5, Build a Gradio Interface

In [ ]:

!pip install gradio --quiet

import gradio as gr
import torch

def predict_mbti(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    conf, pred_id = torch.max(probs, dim=1)
    pred_label = model.config.id2label[pred_id.item()]

    top5 = torch.topk(probs, k=5)
    result = "\n".join([f"{model.config.id2label[i]}: {probs[0][i]:.2f}" for i in top5.indices[0]])

    return f"**Predicted MBTI Type: {pred_label}**\n\nConfidence: {conf.item():.2f}\n\nTop predictions:\n{result}"

interface = gr.Interface(
    fn=predict_mbti,
    inputs=gr.Textbox(lines=6, placeholder="Write a paragraph about yourself"),
    outputs=gr.Markdown(),
    title="MBTI Personality Predictor",
    description="Enter your writing, and the model will guess your MBTI personality type."
)

interface.launch()
